In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT=Path.cwd().parent

matches_path = PROJECT_ROOT / "data" / "raw" / "matches.csv"
deliveries_path = PROJECT_ROOT / "data" / "raw" / "deliveries.csv"

matches_raw = pd.read_csv(matches_path)
deliveries_raw = pd.read_csv(deliveries_path)

matches = matches_raw.copy()
deliveries = deliveries_raw.copy()

print("Datasets loaded successfully!")
print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)

Datasets loaded successfully!
Matches shape: (1095, 20)
Deliveries shape: (260920, 17)


In [2]:
matches_missing = matches.isnull().sum()
deliveries_missing = deliveries.isnull().sum()

matches_missing = matches_missing[matches_missing > 0]
deliveries_missing = deliveries_missing[deliveries_missing > 0]

print("MISSING VALUES — MATCHES")
print(matches_missing.sort_values(ascending=False))

print("\nMISSING VALUES — DELIVERIES")
print(deliveries_missing.sort_values(ascending=False))


MISSING VALUES — MATCHES
method             1074
city                 51
result_margin        19
winner                5
player_of_match       5
target_runs           3
target_overs          3
dtype: int64

MISSING VALUES — DELIVERIES
fielder             251566
player_dismissed    247970
dismissal_kind      247970
extras_type         246795
dtype: int64


In [3]:
matches_missing_pct=(
    matches.isnull().mean()*100
).round(2)

deliveries_missing_pct = (
    deliveries.isnull().mean() * 100
).round(2)

print("MISSING VALUE PERCENTAGES — MATCHES")
print(matches_missing_pct[matches_missing_pct > 0]
      .sort_values(ascending=False))

print("\nMISSING VALUE PERCENTAGES — DELIVERIES")
print(deliveries_missing_pct[deliveries_missing_pct > 0]
      .sort_values(ascending=False))


MISSING VALUE PERCENTAGES — MATCHES
method             98.08
city                4.66
result_margin       1.74
winner              0.46
player_of_match     0.46
target_runs         0.27
target_overs        0.27
dtype: float64

MISSING VALUE PERCENTAGES — DELIVERIES
fielder             96.41
player_dismissed    95.04
dismissal_kind      95.04
extras_type         94.59
dtype: float64


In [4]:
print("MATCHES WITH MISSING WINNER")
print(
    matches.loc[
        matches["winner"].isnull(),
        ["id", "season", "team1", "team2", "result", "result_margin"]
    ].to_string(index=False)
)

print("\nMATCHES WITH A RECORDED METHOD")
print(
    matches.loc[
        matches["method"].notnull(),
        ["id", "season", "city", "method", "result", "winner"]
    ].to_string(index=False)
)

print("\nMATCHES WITH MISSING CITY")
print(
    matches.loc[
        matches["city"].isnull(),
        ["id", "season", "venue", "team1", "team2"]
    ].head(15).to_string(index=False)
)

MATCHES WITH MISSING WINNER
     id season                       team1               team2    result  result_margin
 501265   2011            Delhi Daredevils       Pune Warriors no result            NaN
 829763   2015 Royal Challengers Bangalore    Rajasthan Royals no result            NaN
 829813   2015 Royal Challengers Bangalore    Delhi Daredevils no result            NaN
1178424   2019 Royal Challengers Bangalore    Rajasthan Royals no result            NaN
1359519   2023        Lucknow Super Giants Chennai Super Kings no result            NaN

MATCHES WITH A RECORDED METHOD
     id  season          city method  result                      winner
 336022 2007/08         Delhi    D/L    runs             Kings XI Punjab
 336025 2007/08       Kolkata    D/L    runs         Chennai Super Kings
 392183    2009     Cape Town    D/L wickets            Delhi Daredevils
 392186    2009        Durban    D/L    runs       Kolkata Knight Riders
 392214    2009     Centurion    D/L    runs   

In [5]:
print("METHOD VALUE COUNTS (INCLUDING MISSING)")
print(matches["method"].value_counts(dropna=False))

print("\nMATCHES WITH MISSING RESULT MARGIN")
print(
    matches.loc[
        matches["result_margin"].isnull(),
        ["id", "season", "result", "result_margin", "winner"]
    ].to_string(index=False)
)

print("\nMATCHES WITH MISSING TARGET VALUES")
print(
    matches.loc[
        matches["target_runs"].isnull() |
        matches["target_overs"].isnull(),
        ["id", "season", "result", "target_runs", "target_overs"]
    ].to_string(index=False)
)

print("\nTARGET VALUE MISSINGNESS")
print(
    matches[["target_runs", "target_overs"]]
    .isnull()
    .sum()
)


METHOD VALUE COUNTS (INCLUDING MISSING)
method
NaN    1074
D/L      21
Name: count, dtype: int64

MATCHES WITH MISSING RESULT MARGIN
     id  season    result  result_margin                      winner
 392190    2009       tie            NaN            Rajasthan Royals
 419121 2009/10       tie            NaN             Kings XI Punjab
 501265    2011 no result            NaN                         NaN
 598004    2013       tie            NaN         Sunrisers Hyderabad
 598017    2013       tie            NaN Royal Challengers Bangalore
 729315    2014       tie            NaN            Rajasthan Royals
 829741    2015       tie            NaN             Kings XI Punjab
 829763    2015 no result            NaN                         NaN
 829813    2015 no result            NaN                         NaN
1082625    2017       tie            NaN              Mumbai Indians
1175365    2019       tie            NaN              Delhi Capitals
1178424    2019 no result            Na

In [6]:
missing_city_matches = matches.loc[
    matches["city"].isnull(),
    ["id", "season", "venue"]
]

print("Number of matches with missing city:",
      len(missing_city_matches))

print("\nVENUE COUNTS FOR MATCHES WITH MISSING CITY")
print(
    missing_city_matches["venue"]
    .value_counts(dropna=False)
)

print("\nALL DISTINCT VENUES IN THE DATASET")
print(
    matches["venue"]
    .value_counts(dropna=False)
    .to_string()
)


Number of matches with missing city: 51

VENUE COUNTS FOR MATCHES WITH MISSING CITY
venue
Dubai International Cricket Stadium    33
Sharjah Cricket Stadium                18
Name: count, dtype: int64

ALL DISTINCT VENUES IN THE DATASET
venue
Eden Gardens                                                             77
Wankhede Stadium                                                         73
M Chinnaswamy Stadium                                                    65
Feroz Shah Kotla                                                         60
Rajiv Gandhi International Stadium, Uppal                                49
MA Chidambaram Stadium, Chepauk                                          48
Sawai Mansingh Stadium                                                   47
Dubai International Cricket Stadium                                      46
Wankhede Stadium, Mumbai                                                 45
Punjab Cricket Association Stadium, Mohali                               3

In [7]:
venues_to_check = [
    "Dubai International Cricket Stadium",
    "Sharjah Cricket Stadium"
]

for venue in venues_to_check:
    print(f"\nVenue: {venue}")

    venue_matches = matches.loc[
        matches["venue"] == venue,
        ["season", "city"]
    ]

    print(
        venue_matches["city"]
        .value_counts(dropna=False)
    )



Venue: Dubai International Cricket Stadium
city
NaN      33
Dubai    13
Name: count, dtype: int64

Venue: Sharjah Cricket Stadium
city
NaN        18
Sharjah    10
Name: count, dtype: int64


In [8]:
city_mapping = {
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah"
}

for venue, city in city_mapping.items():
    condition = (
        matches["city"].isnull()
        & (matches["venue"] == venue)
    )

    matches.loc[condition, "city"] = city

print("Missing city values after cleaning:",
      matches["city"].isnull().sum())

print("\nUpdated city counts:")
print(matches.loc[
    matches["venue"].isin(city_mapping.keys()),
    ["venue", "city"]
].groupby(["venue", "city"]).size())


Missing city values after cleaning: 0

Updated city counts:
venue                                city   
Dubai International Cricket Stadium  Dubai      46
Sharjah Cricket Stadium              Sharjah    28
dtype: int64


In [9]:
print("DISMISSAL KIND COUNTS")
print(deliveries["dismissal_kind"].value_counts(dropna=False))


print("\nEXTRAS TYPE COUNTS")
print(deliveries["extras_type"].value_counts(dropna=False))


print("\nDISMISSAL COLUMN MISSINGNESS")
print(
    deliveries[
        ["is_wicket", "player_dismissed", "dismissal_kind", "fielder"]
    ].isnull().sum()
)


print("\nIS_WICKET VALUE COUNTS")
print(deliveries["is_wicket"].value_counts(dropna=False))


DISMISSAL KIND COUNTS
dismissal_kind
NaN                      247970
caught                     8063
bowled                     2212
run out                    1114
lbw                         800
caught and bowled           367
stumped                     358
retired hurt                 15
hit wicket                   15
obstructing the field         3
retired out                   3
Name: count, dtype: int64

EXTRAS TYPE COUNTS
extras_type
NaN        246795
wides        8380
legbyes      4001
noballs      1069
byes          673
penalty         2
Name: count, dtype: int64

DISMISSAL COLUMN MISSINGNESS
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

IS_WICKET VALUE COUNTS
is_wicket
0    247970
1     12950
Name: count, dtype: int64


In [10]:
wicket_missing_details = deliveries.loc[
    (deliveries["is_wicket"] == 1)
    & (
        deliveries["player_dismissed"].isnull()
        | deliveries["dismissal_kind"].isnull()
    )
]

print("Wickets missing dismissal details:",
      len(wicket_missing_details))

non_wicket_with_dismissal = deliveries.loc[
    (deliveries["is_wicket"] == 0)
    & (
        deliveries["player_dismissed"].notnull()
        | deliveries["dismissal_kind"].notnull()
    )
]

print("Non-wicket deliveries with dismissal details:",
      len(non_wicket_with_dismissal))


print("\nEXTRAS TYPE VS EXTRA RUNS")
print(
    deliveries.groupby("extras_type", dropna=False)["extra_runs"]
    .agg(["count", "sum", "min", "max"])
)


missing_extras_type = deliveries.loc[
    (deliveries["extra_runs"] > 0)
    & deliveries["extras_type"].isnull()
]

print("\nDeliveries with extra runs but missing extras_type:",
      len(missing_extras_type))

print(
    missing_extras_type[
        ["match_id", "inning", "over", "ball",
         "batter", "extra_runs", "total_runs"]
    ].head(10).to_string(index=False)
)


Wickets missing dismissal details: 0
Non-wicket deliveries with dismissal details: 0

EXTRAS TYPE VS EXTRA RUNS
              count    sum  min  max
extras_type                         
byes            673   1270    1    5
legbyes        4001   5242    1    5
noballs        1069   1104    1    5
penalty           2     12    5    7
wides          8380  10064    1    5
NaN          246795      0    0    0

Deliveries with extra runs but missing extras_type: 0
Empty DataFrame
Columns: [match_id, inning, over, ball, batter, extra_runs, total_runs]
Index: []


In [11]:
run_columns = ["batsman_runs", "extra_runs", "total_runs"]

print("NEGATIVE RUN VALUES")
for column in run_columns:
    count = (deliveries[column] < 0).sum()
    print(f"{column}: {count}")

incorrect_total_runs = deliveries.loc[
    deliveries["total_runs"]
    != deliveries["batsman_runs"] + deliveries["extra_runs"]
]

print("\nRows where total_runs != batsman_runs + extra_runs:",
      len(incorrect_total_runs))

if not incorrect_total_runs.empty:
    print(
        incorrect_total_runs[
            ["match_id", "inning", "over", "ball",
             "batsman_runs", "extra_runs", "total_runs"]
        ].head(10).to_string(index=False)
    )

print("\nIS_WICKET UNIQUE VALUES")
print(sorted(deliveries["is_wicket"].unique()))

print("\nIS_WICKET VALUES OUTSIDE 0 AND 1:",
      (~deliveries["is_wicket"].isin([0, 1])).sum())

print("\nNEGATIVE OVER VALUES:",
      (deliveries["over"] < 0).sum())

print("NEGATIVE BALL VALUES:",
      (deliveries["ball"] < 0).sum())


NEGATIVE RUN VALUES
batsman_runs: 0
extra_runs: 0
total_runs: 0

Rows where total_runs != batsman_runs + extra_runs: 0

IS_WICKET UNIQUE VALUES
[np.int64(0), np.int64(1)]

IS_WICKET VALUES OUTSIDE 0 AND 1: 0

NEGATIVE OVER VALUES: 0
NEGATIVE BALL VALUES: 0


In [12]:
print("MATCHES DATA TYPES")
print(matches.dtypes)

print("\nDELIVERIES DATA TYPES")
print(deliveries.dtypes)

print("\nMATCHES DATA TYPES SUMMARY")
print(matches.dtypes.value_counts())

print("\nDELIVERIES DATA TYPES SUMMARY")
print(deliveries.dtypes.value_counts())


MATCHES DATA TYPES
id                   int64
season                 str
city                   str
date                   str
match_type             str
player_of_match        str
venue                  str
team1                  str
team2                  str
toss_winner            str
toss_decision          str
winner                 str
result                 str
result_margin      float64
target_runs        float64
target_overs       float64
super_over             str
method                 str
umpire1                str
umpire2                str
dtype: object

DELIVERIES DATA TYPES
match_id            int64
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
batsman_runs        int64
extra_runs          int64
total_runs          int64
extras_type           str
is_wicket           int64
player_dismissed      str
dismissal_kind

In [13]:
team_columns = [
    "team1", "team2", "toss_winner", "winner"
]

for column in team_columns:
    print(f"\n{column.upper()} — UNIQUE VALUES")
    print(sorted(matches[column].dropna().unique()))

print("\nBATTING TEAMS")
print(sorted(deliveries["batting_team"].dropna().unique()))

print("\nBOWLING TEAMS")
print(sorted(deliveries["bowling_team"].dropna().unique()))

print("\nRESULT CATEGORIES")
print(matches["result"].value_counts(dropna=False))

print("\nTOSS DECISIONS")
print(matches["toss_decision"].value_counts(dropna=False))

print("\nMATCH TYPES")
print(matches["match_type"].value_counts(dropna=False))



TEAM1 — UNIQUE VALUES
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

TEAM2 — UNIQUE VALUES
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

TOSS_WINNER — UNIQUE VALUES
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat L

In [14]:
team_columns = [
    "team1", "team2", "toss_winner", "winner"
]

print("TEAM NAME WHITESPACE CHECK")
for column in team_columns:
    mask = matches[column].notna() & (
        matches[column] != matches[column].str.strip()
    )
    print(f"{column}: {mask.sum()} rows with surrounding whitespace")

pune_names = [
    "Rising Pune Supergiant",
    "Rising Pune Supergiants"
]

print("\nPUNE FRANCHISE NAMES BY SEASON")
print(
    matches.loc[
        matches["team1"].isin(pune_names)
        | matches["team2"].isin(pune_names),
        ["season", "team1", "team2"]
    ].groupby(["season", "team1", "team2"]).size()
)


historical_names = [
    "Delhi Daredevils", "Delhi Capitals",
    "Kings XI Punjab", "Punjab Kings",
    "Royal Challengers Bangalore", "Royal Challengers Bengaluru"
]

print("\nHISTORICAL TEAM NAMES BY SEASON")
for name in historical_names:
    seasons = sorted(
        matches.loc[
            (matches["team1"] == name) | (matches["team2"] == name),
            "season"
        ].unique()
    )
    print(f"{name}: {seasons}")


TEAM NAME WHITESPACE CHECK
team1: 0 rows with surrounding whitespace
team2: 0 rows with surrounding whitespace
toss_winner: 0 rows with surrounding whitespace
winner: 0 rows with surrounding whitespace

PUNE FRANCHISE NAMES BY SEASON
season  team1                        team2                      
2016    Delhi Daredevils             Rising Pune Supergiants        1
        Gujarat Lions                Rising Pune Supergiants        1
        Kings XI Punjab              Rising Pune Supergiants        1
        Kolkata Knight Riders        Rising Pune Supergiants        1
        Mumbai Indians               Rising Pune Supergiants        1
        Rising Pune Supergiants      Delhi Daredevils               1
                                     Gujarat Lions                  1
                                     Kings XI Punjab                1
                                     Kolkata Knight Riders          1
                                     Mumbai Indians                 1
 

In [15]:
print("Missing dates:", matches["date"].isna().sum())

parsed_dates = pd.to_datetime(
    matches["date"],
    errors="coerce",
    format="mixed"
)


invalid_dates = parsed_dates.isna() & matches["date"].notna()

print("Invalid non-missing dates:", invalid_dates.sum())


print("Earliest match date:", parsed_dates.min())
print("Latest match date:", parsed_dates.max())

print("\nSample converted dates:")
print(parsed_dates.head())

Missing dates: 0
Invalid non-missing dates: 0
Earliest match date: 2008-04-18 00:00:00
Latest match date: 2024-05-26 00:00:00

Sample converted dates:
0   2008-04-18
1   2008-04-19
2   2008-04-19
3   2008-04-20
4   2008-04-20
Name: date, dtype: datetime64[us]


In [16]:
matches["date"] = pd.to_datetime(
    matches["date"],
    errors="raise",
    format="mixed"
)

print("Date column dtype:", matches["date"].dtype)
print("Missing dates after conversion:", matches["date"].isna().sum())
print("Date range:", matches["date"].min(), "to", matches["date"].max())

Date column dtype: datetime64[us]
Missing dates after conversion: 0
Date range: 2008-04-18 00:00:00 to 2024-05-26 00:00:00


In [17]:
# 1. Verify dataset shapes
print("CLEANED DATASET SHAPES")
print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)

# 2. Check match ID uniqueness
print("\nMATCH ID CHECKS")
print("Unique match IDs:", matches["id"].nunique())
print("Duplicate match IDs:", matches["id"].duplicated().sum())

# 3. Check delivery-to-match consistency
print(
    "Delivery match IDs missing from matches:",
    (~deliveries["match_id"].isin(matches["id"])).sum()
)

# 4. Check key cleaning results
print("\nCLEANING CHECKS")
print("Missing cities after cleaning:", matches["city"].isna().sum())
print("Missing winners:", matches["winner"].isna().sum())
print("Missing match dates:", matches["date"].isna().sum())
print("Invalid wicket indicators:", (~deliveries["is_wicket"].isin([0, 1])).sum())

# 5. Confirm original in-memory copies were preserved
print("\nRAW COPY CHECKS")
print("Original matches city missing:", matches_raw["city"].isna().sum())
print("Original matches date dtype:", matches_raw["date"].dtype)
print("Original deliveries shape:", deliveries_raw.shape)

CLEANED DATASET SHAPES
Matches: (1095, 20)
Deliveries: (260920, 17)

MATCH ID CHECKS
Unique match IDs: 1095
Duplicate match IDs: 0
Delivery match IDs missing from matches: 0

CLEANING CHECKS
Missing cities after cleaning: 0
Missing winners: 5
Missing match dates: 0
Invalid wicket indicators: 0

RAW COPY CHECKS
Original matches city missing: 51
Original matches date dtype: str
Original deliveries shape: (260920, 17)


In [18]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

matches_output = processed_dir / "matches_cleaned.csv"
deliveries_output = processed_dir / "deliveries_cleaned.csv"

matches.to_csv(matches_output, index=False)
deliveries.to_csv(deliveries_output, index=False)

print("Saved files successfully!")
print(matches_output)
print(deliveries_output)


Saved files successfully!
c:\Users\Aryan\Desktop\IPLytics\ipl-data-analysis\data\processed\matches_cleaned.csv
c:\Users\Aryan\Desktop\IPLytics\ipl-data-analysis\data\processed\deliveries_cleaned.csv


# IPLytics — Data Cleaning Report

## 1. Dataset Overview

Two datasets were loaded using Pandas and copied into working DataFrames to preserve the original data.

### Matches Dataset
- Rows: 1,095
- Columns: 20
- Each row represents a match record.
- The date column was converted from a string to a datetime data type.

### Deliveries Dataset
- Rows: 260,920
- Columns: 17
- Each row represents a ball-by-ball delivery record.
- Original delivery records were retained because the integrity checks passed.

## 2. Missing Values Analysis

### Matches Dataset
- Missing city values: 51 initially, reduced to 0 after justified corrections.
- Missing winners: 5, retained because these matches ended with `no result`.
- Missing `method`: 1,074, retained because the values are not applicable to most matches.
- Missing `result_margin`: 19, retained for tied and no-result matches.
- Missing `target_runs` and `target_overs`: 3 each, retained for no-result matches.
- Missing `player_of_match`: 5, retained rather than filled with assumed values.

### Deliveries Dataset
- Missing `player_dismissed` and `dismissal_kind` values were consistent with deliveries that did not result in a wicket.
- Missing `fielder` values were retained because a fielder is not recorded for every delivery.
- Missing `extras_type` values were consistent with deliveries that did not have extras.
- No deliveries with positive `extra_runs` had a missing `extras_type`.

## 3. Data Cleaning and Type Corrections

- Converted the `date` column to a datetime data type after verifying all non-missing dates could be parsed.
- Filled 51 missing city values using verified mappings:
  - `Dubai International Cricket Stadium` → `Dubai`
  - `Sharjah Cricket Stadium` → `Sharjah`
- Checked team names for leading or trailing whitespace; no affected rows were found.
- Preserved historical franchise names, including Delhi Daredevils/Delhi Capitals, Kings XI Punjab/Punjab Kings, and Royal Challengers Bangalore/Royal Challengers Bengaluru.
- Preserved the historical Pune team-name variants instead of assuming they were errors.

## 4. Numeric and Consistency Checks

- Negative values in `batsman_runs`, `extra_runs`, and `total_runs`: 0
- Rows where `total_runs` differed from `batsman_runs + extra_runs`: 0
- Invalid `is_wicket` values outside 0 and 1: 0
- Negative over numbers: 0
- Negative ball numbers: 0
- Wickets missing dismissal details: 0
- Non-wicket deliveries with dismissal details: 0

## 5. Final Validation

- Matches shape: 1,095 rows × 20 columns
- Deliveries shape: 260,920 rows × 17 columns
- Unique match IDs: 1,095
- Duplicate match IDs: 0
- Delivery match IDs absent from matches: 0
- Missing cities after cleaning: 0
- Missing match dates after conversion: 0
- Missing winners retained: 5, corresponding to no-result matches.

All final validation checks passed.

## 6. Processed Data Files

The cleaned datasets were saved separately from the raw data:

- `data/processed/matches_cleaned.csv`
- `data/processed/deliveries_cleaned.csv`

The original files in `data/raw/` were not overwritten. The cleaning process was performed on working copies, preserving the original data for reproducibility.

## 7. Conclusion

The datasets were investigated, legitimate missing values were preserved, justified city corrections were applied, and match dates were converted to a datetime type.

The cleaned datasets passed the final integrity checks and are ready for exploratory data analysis, statistical summaries, and visualizations in the next stage of the project.